### Predator-Prey in the Miramichi River

The Lotka-Volterra model in `predator_prey.ipynb` predicts that predator and
prey populations **cycle forever**: when prey become scarce the predators
starve, and that lets the prey recover.

The Miramichi River in New Brunswick, Canada, is a real system where that
prediction fails. No knowledge of fish or fishing is needed to follow the
story. Everything used later is explained here first.

**The two fish.**

- **Atlantic salmon** (*Salmo salar*) are the **prey**. A salmon hatches in
  the river and lives there for its first few years. It then swims down the
  river and out to the ocean, where it grows to adult size. One or more
  years later it swims back up the same river to breed. The Miramichi was
  once one of the great salmon rivers of the world.
- **Striped bass** (*Morone saxatilis*) are the **predator**. A striped bass
  is a large fish that eats smaller fish. The bass in this story live along
  the coast of the Gulf of St. Lawrence, the sea that the Miramichi flows
  into, and they swim into the lower river each spring to breed.

**Words used in this notebook.**

- **Spawn** means to breed by laying eggs. A **spawner** is an adult fish
  that has come into the river to spawn. Fish populations are usually
  reported as a number of spawners, because the spawners are the fish that
  produce the next generation. Every bass count in this notebook is a count
  of spawners.
- **Smolt.** A young salmon making its first trip from the river to the sea.
  A smolt is small enough for a bass to swallow whole.
- **Returns.** The number of adult salmon that come back from the sea to the
  river in one year. It is also called that year's **run**. Every salmon
  count in this notebook is a count of returning adults.
- **Estuary.** The lower stretch of a river, where fresh river water meets
  salt water from the sea and the water level rises and falls with the tide.
- **Fishery.** The catching of one kind of fish in one area, together with
  the rules for it. To **close** a fishery is to make catching that fish
  illegal. **Overfishing** is catching fish faster than they can breed, so
  the population shrinks.
- **DFO.** Fisheries and Oceans Canada, the department of the Canadian
  government that manages Canada's fisheries. The abbreviation comes from its
  legal name, the Department of Fisheries and Oceans. DFO plays two roles in
  this story. Its scientists count the fish, so the bass and salmon numbers
  quoted here are DFO estimates. It also makes the rules: it decides who may
  catch each kind of fish and how many, and it can close or reopen a fishery.
- **Harvest.** The fish that people catch and keep, which removes them from
  the population. In this notebook the fish being harvested are always
  **striped bass**, never salmon. The people doing the harvesting are anyone
  DFO allows to catch bass: Indigenous (First Nations) communities who catch
  bass to sell, people who fish with a rod for sport, who are called
  **anglers**, and fishing businesses if DFO were to allow them back.
  Harvest is the one part of this story that people control.

**What happened in the Miramichi.**

- **Predator boom.** Overfishing cut the striped bass to about 5,000
  spawners, and DFO closed the bass fishery in 1996. With nobody catching
  them, the bass recovered to about 200,000 spawners by 2011 and 472,000 by
  2022. Most of them spawn in the estuary of the Northwest Miramichi, one of
  the river's two main branches.
- **Prey collapse.** DFO estimates that salmon returns to the Miramichi fell
  by 93% between 2010 and 2024, to fewer than 5,000 returning adults in 2025.
  Salmon returns to nearby rivers without a large bass population did not
  fall.
- **Overlap.** Bass spawn from late May to mid-June in the upper part of the
  Northwest Miramichi estuary. That is exactly when and where the smolts swim
  through on their way to the sea. In the early 2000s about three quarters of
  Northwest Miramichi smolts survived the trip. Today fewer than 5% do.
  In 2023 researchers fitted smolts with small electronic tags that can be
  followed down the river, and about 65% of the tagged smolts were eaten by
  other fish, which in that place and season means mostly bass.
- **No rebound.** Salmon returns have stayed low for more than a decade, and
  the bass have not declined at all.

**The big picture.** This notebook builds a modified Lotka-Volterra model and
uses it to tell one story in four steps.

- **The textbook model has a safety net.** Predators that eat down their prey
  then starve, so the prey always recover and the two populations cycle.
- **The Miramichi has no safety net.** Striped bass live mostly on other
  fish, so they stay abundant whether or not any salmon are left, and nothing
  pushes the salmon back up.
- **The decline feeds itself.** The smaller the salmon run, the more likely
  each remaining salmon is to be eaten.
- **Recovery has a threshold.** Harvesting bass does nothing for the salmon
  until it pushes the bass below a critical level. Past that harvest rate the
  salmon recover.

The model parameters are **illustrative**. They are chosen so the model
follows the real 2000 to 2022 timeline, not fitted to survey data.

---
### Setup: packages and units

Three quantities appear in every equation, plot and printed line.

- **Time $t$** is measured in **years since 2000**, so $t = 0$ is the year
  2000 and $t = 22$ is 2022. Plots show calendar years.
- **Salmon $x$** is one year's returns, written as a fraction of **river
  capacity**: the largest run the river could support if no salmon died at sea
  and there were no bass. $x = 1$ is a river at capacity. The run in the year
  2000 is $x_0 = 0.6$. Plots and tables report salmon as a **percent of the
  year-2000 level**, which is $x / x_0$. So 100% means the run is the size it
  was in 2000, 50% means half that size, and 0% means no salmon came back.
- **Striped bass $y$** is measured in units of **100,000 spawners**, so
  $y = 1$ is 100,000 bass and $y = 5$ is 500,000 bass. Printed lines convert
  $y$ to spawners, and plots show it in thousands of spawners.

Every rate in the notebook is **per year**, which printed lines shorten to
`/yr`. A rate is a fraction of the current population per year: a run growing
at 0.30 per year gains about 30% of its current size each year.

On every plot, **more salmon is the good direction**. The question the
notebook asks of each plot is whether the salmon line goes up, goes down, or
comes back.

In [ ]:
"""miramichi_river.ipynb"""

# Cell 01 - Import packages and set the time axis

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import MultipleLocator
from scipy.integrate import solve_ivp

start_year = 2000  # t = 0 corresponds to this calendar year
bass_unit = 100_000  # one unit of y is this many spawning bass

print(f"t = 0 is the year {start_year}")
print(f"y = 1.0 means {bass_unit:,} striped bass spawners")

---
### The textbook prediction

First, apply the classic Lotka-Volterra equations to salmon $x$ and bass $y$
exactly as in `predator_prey.ipynb`:

$$\frac{dx}{dt} = r x - \beta x y
\qquad
\frac{dy}{dt} = \delta x y - \gamma y$$

- $r x$: salmon grow at net rate $r$ per year when there are no bass
- $-\beta x y$: bass eat salmon in proportion to encounters $x y$
- $\delta x y$: bass grow **only** by eating salmon
- $-\gamma y$: bass die at rate $\gamma$ per year when there are no salmon to
  eat

The last two terms are the key assumption. The bass are **specialists**:
salmon are their only food. If salmon vanish, the bass starve.

The salmon start at the 2000 level $x_0 = 0.6$. The bass start at 5,000
spawners ($y = 0.05$), close to the real population in 2000.

The code cell prints the model's **equilibrium point** $(x^*, y^*)$. This is
the one pair of populations at which both derivatives are zero, so neither
population changes:

$$x^* = \frac{\gamma}{\delta} \qquad y^* = \frac{r}{\beta}$$

Started anywhere else, the populations cycle around this point.

- **Salmon equilibrium $x^*$** is printed as a percent of the starting level
  $x_0$.
- **Bass equilibrium $y^*$** is printed in spawners.
- **Derivatives at $(x^*, y^*)$** are $dx/dt$ and $dy/dt$ evaluated at the
  equilibrium point. Both should be zero.

In [ ]:
# Cell 02 - Classic Lotka-Volterra model and parameters

r = 0.3  # salmon net growth rate with no bass (per year)
beta = 0.12  # salmon loss per salmon-bass encounter
delta = 1.1  # bass gain per salmon-bass encounter
gamma = 0.33  # bass death rate with no salmon (per year)

salmon_0 = 0.6  # starting salmon level (fraction of carrying capacity)
bass_0 = 0.05  # starting bass level (5,000 spawners)


def classic_model(
    time: float,
    state_vector: list[float],
    r: float,
    beta: float,
    delta: float,
    gamma: float,
) -> tuple[float, float]:
    """Return [dx/dt, dy/dt] for the classic Lotka-Volterra system."""
    salmon, bass = state_vector
    d_salmon = r * salmon - beta * salmon * bass
    d_bass = delta * salmon * bass - gamma * bass
    return d_salmon, d_bass


# Equilibrium point: populations cycle around (x*, y*)
salmon_eq = gamma / delta  # x* = gamma/delta
bass_eq = r / beta  # y* = r/beta

print(f"Salmon equilibrium x* : {salmon_eq / salmon_0:.0%} of the starting level")
print(f"Bass equilibrium   y* : {bass_eq * bass_unit:,.0f} spawners")

# Quick check: at the equilibrium both derivatives should be zero
check = classic_model(0, [salmon_eq, bass_eq], r, beta, delta, gamma)
print(f"Derivatives at (x*, y*) = ({check[0]:.1e}, {check[1]:.1e})  (expected 0)")

---
### Running the textbook model

The plotting helper `plot_populations` draws salmon on the top panel and bass
on the bottom panel, sharing one time axis, because the two populations are
measured in different units. Every later model uses the same helper so the
pictures can be compared directly.

- **Salmon returns (% of year 2000)**, top panel: the run as a percent of its
  year-2000 level, $x / x_0 \times 100$.
- **Bass spawners (thousands)**, bottom panel: the number of spawning bass
  divided by 1,000, so 250 on the axis is 250,000 spawners.

A second helper, `callout`, draws a text box with an arrow to one point on a
plot. Every plot in this notebook uses callouts to mark the parts of each
curve that carry the story.

**How to read the plot.** Time runs left to right. In the top panel, higher
means more salmon, and 100% is the size of the run in 2000. In the bottom
panel, higher means more bass. The six numbered callouts mark the steps of
one cycle in the order they happen, so read them 1 through 6, moving between
the two panels.

In [ ]:
# Cell 03 - Solve and plot the classic model

final_time = 60  # years


def plot_populations(
    t: np.ndarray,
    salmon: np.ndarray,
    bass: np.ndarray,
    title: str,
    fig_name: str,
) -> tuple[plt.Axes, plt.Axes]:
    """Plot salmon (percent) above bass (thousands) on a shared year axis."""
    years = start_year + t
    _, (ax_s, ax_b) = plt.subplots(2, 1, num=fig_name, figsize=(10, 6), sharex=True)

    ax_s.plot(
        years,
        salmon / salmon_0 * 100,
        color="blue",
        linewidth=2,
        label="Atlantic salmon (prey)",
    )
    ax_s.set_ylabel("Salmon returns\n(% of year 2000)")
    ax_s.set_title(title)

    ax_b.plot(
        years,
        bass * bass_unit / 1000,
        color="red",
        linewidth=2,
        label="Striped bass (predator)",
    )
    ax_b.set_ylabel("Bass spawners\n(thousands)")
    ax_b.set_xlabel("Year")

    for ax in (ax_s, ax_b):
        ax.xaxis.set_major_locator(MultipleLocator(10))
        ax.xaxis.set_minor_locator(MultipleLocator(1))
        ax.legend(loc="upper right", framealpha=1.0, facecolor="white")
        ax.grid(True, alpha=0.3)
    return ax_s, ax_b


def callout(
    ax: plt.Axes,
    text: str,
    point: tuple[float, float],
    text_at: tuple[float, float],
    color: str = "black",
) -> None:
    """Label one feature of a plot: a text box with an arrow to the point."""
    ax.annotate(
        text,
        xy=point,
        xytext=text_at,
        color=color,
        fontsize=9,
        ha="center",
        va="center",
        arrowprops={"arrowstyle": "->", "color": color},
        bbox={"boxstyle": "round", "facecolor": "white", "edgecolor": color},
    )


sol = solve_ivp(
    classic_model,
    (0, final_time),
    [salmon_0, bass_0],
    max_step=final_time / 3000,
    args=(r, beta, delta, gamma),
)

ax_s, ax_b = plot_populations(
    sol.t,
    sol.y[0],
    sol.y[1],
    "Textbook Lotka-Volterra: bass starve, so salmon always recover",
    "classic_lv",
)
ax_s.yaxis.set_major_locator(MultipleLocator(50))
ax_b.yaxis.set_major_locator(MultipleLocator(250))

# The steps of one cycle, numbered in the order they happen
callout(ax_s, "1. Few bass, so\nsalmon multiply", (2003, 215), (2011, 250), "blue")
callout(ax_b, "2. Plenty to eat, so\nbass boom", (2005.6, 1300), (2014, 1300), "red")
callout(ax_s, "3. Bass eat the\nsalmon down", (2006.5, 120), (2014, 120), "blue")
callout(
    ax_b, "4. Nothing left to eat,\nso bass starve", (2011, 330), (2020, 700), "red"
)
callout(
    ax_s, "5. Bass are gone, so\nsalmon recover", (2033.5, 130), (2025, 200), "blue"
)
callout(ax_b, "6. The cycle\nrepeats", (2036.6, 1200), (2029, 1200), "red")
plt.tight_layout()
plt.show()

---
### What the textbook model predicts

**The story in one sentence.** Predator and prey take turns: every boom
causes the next crash, and every crash sets up the next boom, forever.

**Step by step.** The numbers match the callouts on the plot.

1. **Few bass, so salmon multiply.** In 2000 there are only 5,000 bass, far
   too few to hold the salmon back. By 2004 the run is almost three times its
   year-2000 size.
2. **Plenty to eat, so bass boom.** With that many salmon to eat, the bass
   climb from 5,000 to about 1.5 million by 2006.
3. **Bass eat the salmon down.** That many bass eat salmon far faster than
   the salmon can breed. The run falls to about 1% of its year-2000 size.
4. **Nothing left to eat, so bass starve.** In this model salmon are the only
   thing bass eat, so the bass die off over the next two decades, back down to
   a few thousand.
5. **Bass are gone, so salmon recover.** With almost no bass left, the salmon
   multiply again and reach the same peak in 2035.
6. **The cycle repeats.** The bass boom again in 2037. One full cycle takes
   about 31 years, and each cycle is identical to the one before.

**What to notice.**

- The bass peak comes about two years **after** the salmon peak. The
  predator always follows the prey.
- A salmon collapse always contains the seeds of its own recovery. Every dip
  in the top panel is followed by a rise, because step 4 always happens.
- The sizes are not realistic. No river holds 1.5 million bass. The textbook
  model is here as the baseline to compare against, not as a description of
  the Miramichi.

This model does describe some real systems well. The Stan case study listed in
the sources fits it to records of lynx and snowshoe hare furs traded by the
Hudson's Bay Company from 1900 to 1920, and the fit is good. The reason is
that a lynx eats almost nothing but hares, so it really is a specialist.

Striped bass are not specialists, so step 4 never happens in the Miramichi.

---
### Three ways the Miramichi breaks the textbook model

**Bass are generalists.** Striped bass eat many things besides salmon: other
small fish called gaspereau, smelt, shad and sand lance, and shellfish such
as shrimp and crabs. Salmon are a small part of their diet. Their growth
comes mostly from this alternate food, which gives them their own **carrying
capacity** $L$: the largest bass population that food can sustain, whether or
not there are any salmon. The textbook terms $\delta x y - \gamma y$ are
replaced by

$$\frac{dy}{dt} = \rho\, y\left(1 - \frac{y}{L}\right) + \delta x y - H y$$

- $\rho$ is the bass growth rate on alternate food, per year
- $L$ is the bass carrying capacity, in the same units as $y$ (100,000
  spawners)
- $\delta x y$ is a small bonus from eating salmon
- $-H y$ is a **harvest**: people catch and keep bass at the rate $H$ per
  year

**Predation is depensatory.** A bass can only catch and swallow so many smolts
in the few weeks they pass through the estuary. Ecologists call this a
*Type II* response: each bass's catch levels off when smolts are plentiful,
because a bass that is busy eating one smolt is not catching another.

$$\text{salmon eaten per year} = \frac{\beta x y}{1 + \beta h x}$$

Here $h$ is the handling time, which sets how quickly each bass's catch levels
off. Divide by $x$ to get the risk to an individual salmon,
$\beta y / (1 + \beta h x)$, per year. That risk **grows as salmon get
scarcer**, so a shrinking run is eaten ever faster. Biologists call this
**depensation**: a population that does worse, per individual, the smaller it
gets.

**Salmon also die at sea.** Salmon spend one or more years in the North
Atlantic, where they face warming water and other predators. That is a fixed
marine mortality $m$ per year for each salmon. Salmon also have a carrying
capacity, the river capacity $x = 1$ from the setup section, so their
reproduction is logistic: it slows as the run approaches capacity. With
$\alpha$ the salmon reproduction rate per year, the full salmon equation is

$$\frac{dx}{dt} = \alpha x (1 - x) - m x - \frac{\beta x y}{1 + \beta h x}$$

**Which change matters most?** Both equations contain a logistic term, so
each species competes with itself: $-\alpha x^2$ for salmon and
$-\rho y^2 / L$ for bass. The sustainable fisheries preprint listed in the
sources adds only terms of that kind, $-e x^2$ and $-f y^2$, to the textbook
model. Its closed cycles become paths that settle at a steady state, but the
predator in that model still starves without prey, so no steady state has
the predator present and the prey gone. Self-competition alone stops the
cycling without causing a collapse. The Miramichi collapse comes from the
sign of one term in the bass equation: $+\rho y$ where the textbook has
$-\gamma y$.

---
### Four numbers the equations give before any simulation

The next cell sets the parameters and prints four numbers. Each one follows
from the equations alone.

- **Starting salmon level $x_0$.** With no bass, the salmon equation is
  $\alpha x (1 - x) - m x$, which is zero at $x_0 = 1 - m/\alpha$. This is
  where the run settles when reproduction balances marine mortality, and it is
  the size of the run in 2000. It is printed as a fraction **of river
  capacity**, so 0.60 means 60% of the largest run the river could hold.
- **Critical bass level $y_c$.** When the run is small ($x$ near 0), each
  salmon's net growth rate is $\alpha - m$ and its predation risk is
  $\beta y$. The run shrinks when the risk is the larger of the two, which
  happens when the bass are above $y_c = (\alpha - m)/\beta$. Below $y_c$ a
  small run can grow back. Above it, a small run keeps shrinking. It is
  printed in **spawners**.
- **Bass with no harvest.** With $H = 0$ and few salmon left, the bass
  equation is zero at $y = L$: the bass settle at their carrying capacity. It
  is printed in **spawners**.
- **Critical harvest $H_c$.** With a harvest $H$ and the salmon nearly gone,
  the bass equation is zero at $y = L(1 - H/\rho)$, so more harvest means
  fewer bass. Setting that equal to $y_c$ gives the smallest harvest that
  holds the bass down at the critical level, $H_c = \rho(1 - y_c/L)$. It is
  printed **per year**.

**Converting a harvest rate to a percent.** $H$ is a rate applied
continuously through the year. Bass removed at the rate $H$ for one year
shrink by the factor $e^{-H}$, so the fraction removed in that year is
$1 - e^{-H}$. The cell prints this next to $H_c$ as the **percent of the bass
removed each year**.

In [ ]:
# Cell 04 - Miramichi model parameters

alpha = 0.5  # salmon reproduction rate in the river (per year)
m = 0.2  # salmon marine mortality (per year)
# beta = 0.12 is reused from the textbook model: the same attack rate
h = 10.0  # handling time: how quickly each bass's smolt catch saturates

rho = 0.35  # bass growth rate on alternate food (per year)
L = 5.0  # bass carrying capacity set by alternate food (500,000 spawners)
delta_g = 0.05  # small bonus to bass growth from eating salmon

# With no bass, salmon settle where reproduction balances marine mortality
salmon_0 = 1 - m / alpha  # x0 = 1 - m/alpha

# Critical bass level: above it, a small salmon run shrinks instead of growing
bass_crit = (alpha - m) / beta  # y_c = (alpha - m)/beta

# Critical harvest: the smallest H that holds the bass below y_c
harvest_crit = rho * (1 - bass_crit / L)  # H_c = rho(1 - y_c/L)

print(f"Starting salmon level x0 : {salmon_0:.2f} of river capacity")
print(f"Critical bass level y_c  : {bass_crit * bass_unit:,.0f} spawners")
print(f"Bass with no harvest     : {L * bass_unit:,.0f} spawners")
print(
    f"Critical harvest H_c     : {harvest_crit:.2f} per year "
    f"({1 - np.exp(-harvest_crit):.0%} of the bass removed each year)"
)

---
### Coding the Miramichi model

The harvest is switched on at a chosen year `harvest_start`, so the same
function can model the past (no harvest) and a future management policy.

The cell ends with two quick checks that print derivatives.

- $dx/dt$ is the change in the salmon run, in fractions of river capacity per
  year.
- $dy/dt$ is the change in the bass, in units of 100,000 spawners per year. A
  value of 0.1 would mean the bass are growing by 10,000 spawners a year.
- **With no bass**, the salmon start at $x_0$ and there are no bass to grow,
  so both derivatives should be zero.
- **With no salmon and 100k bass** means $x = 0$ and $y = 1$, which is 100,000
  spawners. A positive $dy/dt$ shows that the bass grow with no salmon to eat
  at all, which the textbook model does not allow.

In [ ]:
# Cell 05 - Miramichi ODE system
# state_vector = [salmon, bass]; returns [d_salmon/dt, d_bass/dt]


def miramichi_model(
    time: float,
    state_vector: list[float],
    harvest: float = 0.0,
    harvest_start: float = np.inf,
) -> tuple[float, float]:
    """Return [dx/dt, dy/dt] for salmon and generalist striped bass."""
    salmon, bass = state_vector
    eaten = beta * salmon * bass / (1 + beta * h * salmon)  # Type II
    H = harvest if time >= harvest_start else 0.0

    d_salmon = alpha * salmon * (1 - salmon) - m * salmon - eaten
    d_bass = rho * bass * (1 - bass / L) + delta_g * salmon * bass - H * bass
    return d_salmon, d_bass


# Quick check: with no bass, salmon should sit at x0 and not change
d_s, d_b = miramichi_model(0, [salmon_0, 0.0])
print(f"With no bass: dx/dt = {d_s:.1e}, dy/dt = {d_b:.1e}  (expected 0, 0)")

# Quick check: with no salmon, bass should grow on alternate food alone
d_s, d_b = miramichi_model(0, [0.0, 1.0])
print(f"With no salmon and 100k bass: dy/dt = {d_b:.3f}  (positive, bass grow)")

---
### Depensation: the predator pit

For a salmon run to grow, each salmon's chance of reproducing must beat its
chance of dying. A **per-capita** rate is a rate for one individual salmon,
per year: the run's total rate of change divided by the size of the run.
Plotting two per-capita rates against the size of the salmon run shows when
the run grows and when it shrinks.

- **Salmon reproduction minus marine mortality** (blue line) is each salmon's
  net growth rate with no bass, $\alpha(1 - x) - m$.
- **Predation risk** (red curves) is each salmon's chance of being eaten,
  $\beta y / (1 + \beta h x)$. There is one red curve for each of three bass
  populations: 100,000 (dotted), 250,000 (dashed) and 500,000 (solid)
  spawners.

**How to read the plot.** This plot has no time axis. The horizontal axis is
the size of the salmon run as a percent of its year-2000 level, so the left
edge is a run that is nearly extinct and the right edge is the run as it was
in 2000. The vertical axis is a rate per year. At any run size, compare the
height of the blue line with the height of one red curve:

- Blue above red is **good for salmon**: growth beats predation and the run
  gets bigger, which moves it to the right.
- Red above blue is **bad for salmon**: predation beats growth and the run
  gets smaller, which moves it to the left.

The callouts mark what that comparison says for each of the three bass
populations. The green dot marks the one place where a red curve crosses the
blue line.

The lines printed under the plot give the two rates at the left edge for a
nearly extinct run facing 500,000 bass, and the run size at the green dot.
Rates are per-capita rates per year, printed as `/yr`.

In [ ]:
# Cell 06 - Per-capita salmon growth versus predation risk

salmon_grid = np.linspace(0, salmon_0, 400)
salmon_pct = salmon_grid / salmon_0 * 100  # run size as a percent of year 2000
growth = alpha * (1 - salmon_grid) - m

plt.figure("depensation", figsize=(10, 5.5))
plt.plot(
    salmon_pct,
    growth,
    color="blue",
    linewidth=2,
    label="Salmon reproduction minus marine mortality",
)

for bass_level, style in [(1.0, ":"), (2.5, "--"), (5.0, "-")]:
    risk = beta * bass_level / (1 + beta * h * salmon_grid)
    plt.plot(
        salmon_pct,
        risk,
        color="red",
        linewidth=2,
        linestyle=style,
        label=f"Predation risk, {bass_level * bass_unit:,.0f} bass",
    )

# With 100,000 bass the two rates balance where the dotted curve crosses blue
risk_100k = beta * 1.0 / (1 + beta * h * salmon_grid)
i_bal = np.argmin(np.abs(growth - risk_100k))
plt.plot(salmon_pct[i_bal], growth[i_bal], "o", color="green", markersize=10)

plt.title("Per-capita rates: fewer salmon means each one is more likely eaten")
plt.xlabel("Salmon returns (% of year 2000)")
plt.ylabel("Rate (per year)")

ax = plt.gca()
callout(
    ax,
    "500,000 bass: red is above blue at\nevery run size, so the run shrinks to zero",
    (57, 0.425),
    (76, 0.485),
    "red",
)
callout(
    ax,
    "250,000 bass, the critical level:\nthe two rates meet at zero salmon",
    (0, 0.30),
    (22, 0.37),
    "red",
)
callout(
    ax,
    "100,000 bass: blue is above red here,\nso a small run grows",
    (12, 0.17),
    (26, 0.035),
    "blue",
)
callout(
    ax,
    f"With 100,000 bass the run\nsettles here, at {salmon_pct[i_bal]:.0f}%",
    (salmon_pct[i_bal], growth[i_bal]),
    (87, 0.275),
    "green",
)
ax.xaxis.set_major_locator(MultipleLocator(20))
ax.xaxis.set_minor_locator(MultipleLocator(5))
ax.yaxis.set_major_locator(MultipleLocator(0.1))
ax.set_ylim(0, 0.68)
ax.legend(loc="upper right", framealpha=1.0, facecolor="white")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

risk_500k = beta * L
print(f"With 500,000 bass, a nearly extinct run faces predation {risk_500k:.2f}/yr")
print(f"but can only grow at {alpha - m:.2f}/yr, so it keeps shrinking.")
print(f"With 100,000 bass, the run settles at {salmon_pct[i_bal]:.0f}% of year 2000.")

---
### Reading the depensation plot

**The story in one sentence.** Whether a small salmon run can grow back
depends entirely on how many bass there are, and with too many bass the run
shrinks faster the smaller it gets.

**Curve by curve.** Each item matches a callout on the plot.

- **100,000 bass (dotted).** On the left of the plot the blue line is above
  the dotted curve, so a small run grows and moves right. On the far right
  the dotted curve is above the blue line, so a large run shrinks and moves
  left. Both sides push the run toward the **green dot**, where the two rates
  are equal. With 100,000 bass the run settles there, at 74% of its year-2000
  size, and stays. The salmon and the bass can live together.
- **250,000 bass (dashed), the critical level.** The dashed curve touches the
  blue line only at the left edge, at zero salmon, and is above it everywhere
  else. This is the critical bass level $y_c = (\alpha - m)/\beta$ from Cell
  04: the largest bass population a small run can survive.
- **500,000 bass (solid).** The solid curve is above the blue line at
  **every** run size. There is no place the run can hold steady, so it slides
  all the way to the left edge. At that edge each salmon is eaten at 0.60 per
  year but the run can only grow at 0.30 per year.

**What to notice.** Every red curve rises toward the left. A small run is
eaten faster per fish than a large one, which is the opposite of the textbook
model, where each salmon's risk does not depend on how many salmon there are.
That rising risk is the **predator pit**: the further the run falls, the
harder it is to climb back out.

---
### Simulating the Miramichi from 2000

The next cell runs the Miramichi model for 40 years with no harvest. It starts
in 2000 with the salmon run at $x_0$ and 5,000 bass spawners.

**How to read the plot.** It has the same layout as the textbook plot: time
runs left to right, salmon are on top and bass are below. Higher in the top
panel is good for salmon. Start with the bottom panel, because in this model
the bass drive everything, and then look at what the salmon do in the same
years.

- **Critical level** (dotted red line, bottom panel) is the critical bass
  level $y_c$ = 250,000 spawners from Cell 04. Once the bass curve is above
  this line, a small salmon run can no longer grow back.
- **DFO reference point** (dashed purple line, bottom panel) is DFO's *limit
  reference point* for the bass, 330,000 spawners. A limit reference point is
  the population size below which DFO considers a fish population to be
  seriously harmed. DFO sets its fishing rules to keep the population above
  it.
- **DFO bass estimates** (black diamonds, bottom panel) are DFO's counts of
  the real bass in the river: about 200,000 in 2011 and 472,000 in 2022. They
  are the only real data on the plot. Everything else is the model.
- **93% decline** (dotted blue line, top panel) marks a salmon run at 7% of
  its year-2000 level, which is a fall of 93%. DFO estimates a decline of that
  size between 2010 and 2024. The line measures the fall from 2000, when the
  model's run is somewhat larger than in 2010, so it is a guide for the eye
  and the text after the plot makes the exact 2010 to 2024 comparison.

The table under the plot lists the model's populations in a few milestone
years.

- **year** is the calendar year.
- **salmon (% of year 2000)** is the salmon run as a percent of its year-2000
  level.
- **bass spawners** is the number of spawning bass, rounded to the nearest
  thousand.

In [ ]:
# Cell 07 - Simulate the Miramichi from 2000 with no harvest

final_time = 40  # years: 2000 to 2040
bass_0 = 0.05  # 5,000 spawners in 2000
bass_lrp = 3.3  # DFO limit reference point: 330,000 spawners

sol = solve_ivp(
    miramichi_model,
    (0, final_time),
    [salmon_0, bass_0],
    max_step=final_time / 2000,
)
t = sol.t
salmon, bass = sol.y

ax_s, ax_b = plot_populations(
    t,
    salmon,
    bass,
    "Miramichi model: generalist bass stay high and salmon do not recover",
    "miramichi_no_harvest",
)
ax_b.axhline(
    bass_crit * bass_unit / 1000,
    color="red",
    linestyle=":",
    alpha=0.6,
    label=f"Critical level ({bass_crit * bass_unit:,.0f})",
)
ax_b.axhline(
    bass_lrp * bass_unit / 1000,
    color="purple",
    linestyle="--",
    alpha=0.6,
    label=f"DFO reference point ({bass_lrp * bass_unit:,.0f})",
)
# DFO's counts of real bass in the river: 200,000 in 2011 and 472,000 in 2022
ax_b.plot(
    [2011, 2022],
    [200, 472],
    "D",
    color="black",
    markersize=7,
    label="DFO bass estimates (2011, 2022)",
)
ax_s.axhline(7, color="blue", linestyle=":", alpha=0.6, label="93% decline")

# The year the bass pass the critical level
i_crit = np.argmax(bass > bass_crit)
year_crit = start_year + t[i_crit]

callout(
    ax_b,
    f"Bass pass the critical\nlevel in {year_crit:.0f}",
    (year_crit, bass_crit * bass_unit / 1000),
    (2005, 400),
    "red",
)
callout(
    ax_b,
    "Bass level off near 500,000\nand never fall back",
    (2030, 500),
    (2031, 390),
    "red",
)
callout(ax_s, "Salmon slide as\nthe bass climb", (2015.5, 42), (2008, 35), "blue")
callout(
    ax_s,
    "No rebound: nothing in the\nmodel brings the salmon back",
    (2032, 1),
    (2031, 40),
    "blue",
)
ax_s.yaxis.set_major_locator(MultipleLocator(20))
ax_b.yaxis.set_major_locator(MultipleLocator(100))
ax_s.legend(loc="upper right", framealpha=1.0, facecolor="white")
ax_b.legend(loc="lower right", framealpha=1.0, facecolor="white")
plt.tight_layout()
plt.show()

# Table of the model at a few milestone years
milestones = [2000, 2005, 2010, 2011, 2015, 2020, 2022, 2024]
idx = [np.argmin(np.abs(start_year + t - yr)) for yr in milestones]
pd.DataFrame(
    {
        "year": milestones,
        "salmon (% of year 2000)": np.round(salmon[idx] / salmon_0 * 100, 1),
        "bass spawners": np.round(bass[idx] * bass_unit, -3).astype(int),
    }
)

---
### What the Miramichi model shows

**The story in one sentence.** The bass rise and stay up, the salmon fall and
stay down, and nothing in the model ever reverses either one.

**Step by step.** The years and numbers are in the table under the plot.

1. **2000 to 2010: the bass build up.** Living on their other food, the bass
   grow from 5,000 to 157,000. The salmon barely notice: the run is still at
   82% of its year-2000 size in 2010.
2. **2011: the model meets the real count.** The model has 202,000 bass. The
   first black diamond, DFO's count of about 200,000, sits on the red curve.
3. **2012: the bass pass the critical level.** This is the callout on the
   bottom panel. From here on, a small salmon run cannot grow back.
4. **2012 to 2024: the salmon slide.** The run is at 45% in 2015, 14% in 2020
   and 4.5% in 2024. Between 2010 and 2024 the model's run falls by about
   94%, close to DFO's 93% for the same years.
5. **2022: the bass level off.** The model has 494,000 bass, near their
   carrying capacity of 500,000. The second black diamond, DFO's count of
   472,000, sits just under the curve.
6. **After 2024: no rebound.** The bass stay flat at 500,000 and the salmon
   stay near zero for the rest of the plot.

**What to notice.**

- Compare the bottom panel with the textbook plot. There the bass crashed
  after eating the salmon down. Here the bass **never decline**, because the
  salmon were never what fed them. So the salmon's fall in the top panel has
  no rise after it.
- The dashed purple line, DFO's limit reference point of 330,000, is **above**
  the dotted red critical level of 250,000. In this model any bass population
  above 250,000 drives the salmon toward zero, so a bass population kept
  above its reference point leaves no room for the salmon to recover.

---
### Comparing the two models in the phase plane

A phase plane has **no time axis**. It plots one population against the
other, so each moment in time is a single point and a whole simulation
becomes a path. The next cell draws three paths for each model, textbook on
the left and Miramichi on the right.

**How to read either panel.**

- **Salmon returns (% of year 2000)** runs across. Moving **left** means
  fewer salmon, which is the bad direction for salmon. The left edge, 0%, is
  a river with no salmon. 100% is the run as it was in the year 2000.
- **Bass spawners (thousands)** runs up. Moving **up** means more bass.
- **The dots** mark where each path starts, in the year 2000. All three start
  with the salmon at 100%. They differ only in how many bass there are at the
  start, which the legend gives as **Bass start**: 5,000, 50,000 or 150,000
  spawners.
- **The arrowheads** on each path show which way time moves along it. Follow
  a path from its dot in the direction of its arrowheads.

The line printed under the plot is the **end state** of the last Miramichi
path (bass start 150,000) after 60 years: the salmon run as a percent of its
year-2000 level, and the bass in spawners.

In [ ]:
# Cell 08 - Phase plane: textbook loops versus the Miramichi model


def direction_arrows(
    ax: plt.Axes,
    x: np.ndarray,
    y: np.ndarray,
    t: np.ndarray,
    times: list[float],
    color: str,
) -> None:
    """Draw arrowheads on a path to show which way time moves along it."""
    for time in times:
        i = np.searchsorted(t, time)
        ax.annotate(
            "",
            xy=(x[i + 1], y[i + 1]),
            xytext=(x[i], y[i]),
            arrowprops={"arrowstyle": "-|>", "color": color, "mutation_scale": 22},
        )


_, (ax_c, ax_m) = plt.subplots(1, 2, num="phase_plane", figsize=(11, 5.5))

for bass_start in [0.05, 0.5, 1.5]:
    label = f"Bass start {bass_start * bass_unit:,.0f}"

    # Textbook model: a dot at the year-2000 start, arrowheads at 4 later times
    run = solve_ivp(
        classic_model,
        (0, 60),
        [salmon_0, bass_start],
        max_step=0.02,
        args=(r, beta, delta, gamma),
    )
    x_pct = run.y[0] / salmon_0 * 100
    y_k = run.y[1] * bass_unit / 1000
    (line,) = ax_c.plot(x_pct, y_k, linewidth=2, label=label)
    ax_c.plot(x_pct[0], y_k[0], "o", color=line.get_color(), markeredgecolor="black")
    direction_arrows(ax_c, x_pct, y_k, run.t, [2, 5, 7, 12], line.get_color())

    # Miramichi model: the same start, arrowheads at 3 later times
    run = solve_ivp(miramichi_model, (0, 60), [salmon_0, bass_start], max_step=0.02)
    x_pct = run.y[0] / salmon_0 * 100
    y_k = run.y[1] * bass_unit / 1000
    (line,) = ax_m.plot(x_pct, y_k, linewidth=2, label=label)
    ax_m.plot(x_pct[0], y_k[0], "o", color=line.get_color(), markeredgecolor="black")
    direction_arrows(ax_m, x_pct, y_k, run.t, [6, 11, 16], line.get_color())

ax_m.plot(
    0,
    L * bass_unit / 1000,
    "o",
    color="red",
    markersize=10,
    label="No salmon, bass at capacity",
)

callout(ax_c, "The three paths start\nat these dots, in 2000", (100, 150), (200, 420))
callout(ax_c, "Each loop returns to\nits start, then repeats", (185, 1030), (245, 1150))
callout(ax_m, "The three paths start\nat these dots, in 2000", (100, 50), (70, 60))
callout(ax_m, "Every path ends here\nand stays", (0, 500), (22, 330))

ax_c.set_title("Textbook Lotka-Volterra: closed cycles")
ax_m.set_title("Miramichi model: every path ends at zero salmon")
for ax in (ax_c, ax_m):
    ax.set_xlabel("Salmon returns (% of year 2000)")
    ax.set_ylabel("Bass spawners (thousands)")
    ax.legend(loc="upper right", framealpha=1.0, facecolor="white")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(
    f"Miramichi model end state: salmon {run.y[0, -1] / salmon_0:.2%}, "
    f"bass {run.y[1, -1] * bass_unit:,.0f}"
)

---
### What the phase plane shows

**The story in one sentence.** In the textbook model every path is a round
trip, and in the Miramichi model every path is a one-way trip to a river with
no salmon.

**Textbook panel (left).** Follow the blue path from its dot at the bottom.
The arrowheads take it **counterclockwise**, through the same steps as the
numbered callouts on the textbook time plot:

1. **Right** along the bottom: few bass, so the salmon multiply.
2. **Up** the right side: plenty to eat, so the bass boom.
3. **Left** across the top: the bass eat the salmon down.
4. **Down** the left side: nothing left to eat, so the bass starve.
5. **Right** again, back to the dot it started from.

The left side of the loop is the bad place for salmon, but no path stays
there. Each path is a **closed loop**, which is what a cycle that repeats
forever looks like when time is not drawn. Starting with more bass gives a
smaller loop (orange, then green), but it is still a loop.

**Miramichi panel (right).** Each path starts at its dot on the right edge,
with the salmon at 100%. The arrowheads take it **up and to the left**: the
bass rise while the salmon fall. So the curves are read from right to left,
not left to right, and a curve that looks like it is "going down" is really
the salmon disappearing as the bass climb.

- No path turns back. There is no step 4, because the bass never starve.
- All three paths end at the **red dot** in the upper left corner: 0% salmon
  and 500,000 bass spawners, which is the bass carrying capacity $L$.
- The printed end state confirms it for the last path: salmon 0.00%, bass
  500,000.

**What to notice.** How many bass there were in 2000 makes no difference to
where the Miramichi paths end. The red dot is a **stable equilibrium**: once
the populations arrive they stay, and a small push away from the dot leads
straight back to it. It is the worst outcome for salmon, and the textbook
model has no such point.

---
### Can harvesting bass bring the salmon back?

Harvesting here means people catching and keeping **striped bass**, so that
fewer bass are left in the river to eat smolts. No salmon are harvested in
any scenario. The Atlantic Salmon Federation, a conservation group, and
others have asked DFO to allow a larger bass harvest and bring the spawning
population down to about **100,000**. The model can test that idea.

Each scenario below runs the no-harvest model to 2025, then turns on a
constant harvest rate $H$ per year. The model does not say who catches the
bass. It only says how fast they are removed. The percent of the bass removed
each year is $1 - e^{-H}$, as in Cell 04.

- **No harvest** (black), $H = 0$
- **Modest harvest** (orange), $H = 0.15$ per year (14% of the bass each
  year), below the critical harvest $H_c$
- **Target harvest** (green), $H = 0.30$ per year (26% of the bass each
  year), which holds the bass near 100,000 spawners

**How to read the plot.** Time runs left to right from 2000 to 2100, with
salmon on top and bass below.

- The **dotted gray vertical line** marks 2025, when the harvest begins. To
  the left of it the three scenarios are identical, so the three curves lie
  on top of each other and only the last one drawn is visible.
- The **dotted red line** across the bottom panel is the critical bass level
  from Cell 04, 250,000 spawners.
- To the right of 2025, look at the bottom panel first to see what each
  harvest does to the bass, and then at the top panel to see what that does
  for the salmon. A salmon curve that climbs back toward 100% is a recovery.
  A curve that stays flat along the bottom is no recovery.

The table under the plot gives each scenario's outcome.

- **scenario** is the name of the harvest policy.
- **H (per year)** is its harvest rate.
- **salmon in 2100 (%)** is the salmon run in 2100 as a percent of its
  year-2000 level.
- **bass in 2100** is the number of bass spawners in 2100, rounded to the
  nearest thousand.

In [ ]:
# Cell 09 - Harvest scenarios starting in 2025

final_time = 100  # years: 2000 to 2100
harvest_start = 25  # harvest begins in 2025

scenarios = [
    ("No harvest", 0.0, "black"),
    ("Modest harvest", 0.15, "orange"),
    ("Target harvest", 0.30, "green"),
]

_, (ax_s, ax_b) = plt.subplots(2, 1, num="harvest", figsize=(10, 6.5), sharex=True)
summary = []
for name, H, color in scenarios:
    run = solve_ivp(
        miramichi_model,
        (0, final_time),
        [salmon_0, bass_0],
        max_step=0.05,
        args=(H, harvest_start),
    )
    years = start_year + run.t
    label = f"{name} (H = {H:.2f}/yr)"
    ax_s.plot(years, run.y[0] / salmon_0 * 100, color=color, linewidth=2, label=label)
    ax_b.plot(years, run.y[1] * bass_unit / 1000, color=color, linewidth=2, label=label)
    summary.append((name, H, run.y[0, -1] / salmon_0 * 100, run.y[1, -1] * bass_unit))

# The critical bass level from Cell 04, drawn across the bass panel
ax_b.axhline(bass_crit * bass_unit / 1000, color="red", linestyle=":")
ax_b.text(
    2100,
    232,
    f"Critical bass level ({bass_crit * bass_unit:,.0f})",
    color="red",
    fontsize=9,
    ha="right",
)

for ax in (ax_s, ax_b):
    ax.axvline(start_year + harvest_start, color="gray", linestyle=":")
    ax.xaxis.set_major_locator(MultipleLocator(10))
    ax.xaxis.set_minor_locator(MultipleLocator(2))
    ax.grid(True, alpha=0.3)

callout(ax_s, "Harvest begins\nin 2025", (2025, 70), (2036, 85), "gray")
callout(ax_s, "Target harvest:\nsalmon recover", (2057, 52), (2046, 70), "green")
callout(
    ax_s,
    "No harvest and modest harvest:\nsalmon stay near zero",
    (2078, 0.5),
    (2080, 30),
)
callout(
    ax_b,
    "Modest harvest: fewer bass, but still\nabove the critical level",
    (2050, 287),
    (2060, 390),
    "darkorange",
)
callout(
    ax_b,
    "Target harvest: bass held\nbelow the critical level",
    (2050, 98),
    (2066, 45),
    "green",
)
ax_s.legend(loc="upper right", framealpha=1.0, facecolor="white")
ax_s.set_title("Harvesting bass from 2025: only a large enough harvest works")
ax_s.set_ylabel("Salmon returns\n(% of year 2000)")
ax_b.set_ylabel("Bass spawners\n(thousands)")
ax_b.set_xlabel("Year")
ax_s.yaxis.set_major_locator(MultipleLocator(20))
ax_b.yaxis.set_major_locator(MultipleLocator(100))
plt.tight_layout()
plt.show()

table = pd.DataFrame(
    summary, columns=["scenario", "H (per year)", "salmon in 2100 (%)", "bass in 2100"]
)
table["salmon in 2100 (%)"] = table["salmon in 2100 (%)"].round(1)
table["bass in 2100"] = table["bass in 2100"].round(-3).astype(int)
table

---
### What the harvest scenarios show

**The story in one sentence.** Catching some bass is not enough: the salmon
come back only if the harvest pushes the bass below the critical level.

**Scenario by scenario.** Each item matches a callout on the plot.

- **No harvest (black).** Nothing changes. The bass stay at 500,000 and the
  salmon stay at zero through 2100.
- **Modest harvest (orange).** The bass fall quickly, from 500,000 to about
  286,000. That is a cut of more than 40%, and it does **nothing** for the
  salmon: the run in 2100 is 0.1% of its year-2000 size. The reason is on the
  bottom panel. The orange curve levels off **above** the dotted red critical
  level, so a small salmon run still cannot grow.
- **Target harvest (green).** The bass drop below the critical level within
  about four years and level off near 100,000. The salmon respond slowly.
  The run bottoms out at about 2% around 2029, is back to 27% by 2050 and 60%
  by 2060, and levels off at 73% of its year-2000 size.

**What to notice.**

- **Recovery is slow.** Even with the target harvest, the salmon need about
  35 years to get most of the way back, because a tiny run can only grow so
  fast.
- **Recovery is not complete.** The run levels off at 73%, not 100%, because
  the 100,000 bass that remain still eat salmon. This is the green dot on the
  depensation plot, where the run settles with 100,000 bass.
- **The harvest must continue.** The green curve stays up only because the
  harvest goes on every year. Stop the harvest and the bass return to 500,000.

---
### A tipping point in harvest rate

The three scenarios suggest a threshold somewhere between a harvest rate of
0.15 and 0.30 per year. Sweeping $H$ across many values and recording where
the salmon end up shows that threshold directly. Each point runs the model
to 2125, a century after the harvest begins. The populations in 2125 are
called the **long-run** values.

**How to read the plot.** This plot has no time axis. The horizontal axis is
the harvest rate $H$ per year, so each position across the plot is a
different policy, and the curves show where that policy leaves the two
populations in 2125. Moving right means catching bass faster.

- **Salmon (left axis)**, the blue curve, is the long-run salmon run as a
  percent of its year-2000 level. Read its height on the left axis. Higher is
  better for salmon.
- **Bass (right axis)**, the red curve, is the long-run number of bass in
  thousands of spawners. Read its height on the right axis.
- **Critical harvest $H_c$**, the dashed gray line, is the threshold computed
  in Cell 04.
- **The three large dots** on the blue curve are the three scenarios from the
  previous plot, in the same colors: no harvest (black), modest harvest
  (orange) and target harvest (green).

The line printed under the plot repeats $H_c$ per year and as the percent of
the bass removed each year.

In [ ]:
# Cell 10 - Long-run salmon level versus harvest rate

harvest_rates = np.linspace(0, 0.40, 81)
final_salmon = []
final_bass = []
for H in harvest_rates:
    run = solve_ivp(
        miramichi_model,
        (0, 125),
        [salmon_0, bass_0],
        max_step=0.1,
        args=(H, harvest_start),
    )
    final_salmon.append(run.y[0, -1] / salmon_0 * 100)
    final_bass.append(run.y[1, -1] * bass_unit / 1000)

_, ax_s = plt.subplots(num="harvest_sweep", figsize=(10, 5.5))
(salmon_line,) = ax_s.plot(
    harvest_rates, final_salmon, color="blue", linewidth=2, label="Salmon (left axis)"
)
crit_line = ax_s.axvline(
    harvest_crit,
    color="gray",
    linestyle="--",
    label=f"Critical harvest H_c = {harvest_crit:.2f}/yr",
)
ax_s.set_xlabel("Bass harvest rate H (per year)")
ax_s.set_ylabel("Long-run salmon returns (% of year 2000)")
ax_s.set_title("Salmon recover only when the harvest passes a critical rate")

ax_b = ax_s.twinx()
(bass_line,) = ax_b.plot(
    harvest_rates, final_bass, color="red", linewidth=2, label="Bass (right axis)"
)
ax_b.set_ylabel("Long-run bass spawners (thousands)")

# Mark the three scenarios from Cell 09 on the salmon curve
for name, H, color in scenarios:
    ax_s.plot(
        H,
        np.interp(H, harvest_rates, final_salmon),
        "o",
        color=color,
        markersize=10,
        markeredgecolor="black",
        zorder=5,
    )
target_salmon = np.interp(0.30, harvest_rates, final_salmon)

callout(ax_s, "No harvest", (0.0, 0), (0.03, 14))
callout(ax_s, "Modest harvest:\nstill no salmon", (0.15, 0), (0.105, 22), "darkorange")
callout(
    ax_s,
    f"Target harvest:\nsalmon at {target_salmon:.0f}%",
    (0.30, target_salmon),
    (0.345, 52),
    "green",
)
callout(
    ax_s,
    "Tipping point: past this rate\nthe salmon start to recover",
    (harvest_crit, 32),
    (0.075, 45),
    "gray",
)

ax_s.xaxis.set_major_locator(MultipleLocator(0.05))
ax_s.xaxis.set_minor_locator(MultipleLocator(0.01))
ax_s.yaxis.set_major_locator(MultipleLocator(20))
ax_b.yaxis.set_major_locator(MultipleLocator(100))
lines = [salmon_line, crit_line, bass_line]
ax_s.legend(
    lines,
    [ln.get_label() for ln in lines],
    loc="upper center",
    framealpha=1.0,
    facecolor="white",
)
ax_s.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(
    f"Critical harvest H_c = {harvest_crit:.2f}/yr, about "
    f"{1 - np.exp(-harvest_crit):.0%} of the bass each year"
)

---
### What the harvest sweep shows

**The story in one sentence.** The salmon do not come back a little for a
little harvest: they stay at zero until the harvest rate passes a tipping
point, and only then begin to recover.

**Left to right along the blue curve.**

- **From 0 to about 0.17 per year the curve is flat at zero.** Every policy in
  this range leaves the river with no salmon. The black dot (no harvest) and
  the orange dot (modest harvest) are both on this flat stretch, which is why
  the modest harvest achieved nothing.
- **At the dashed gray line the curve lifts off.** This is the critical
  harvest $H_c$, the tipping point marked by the callout. It is the harvest
  rate that brings the bass down to exactly their critical level of 250,000,
  which can be read where the red curve crosses the dashed line.
- **Past the line the curve climbs steeply.** The long-run salmon run is 18%
  of its year-2000 size at a harvest rate of 0.20, 52% at 0.25, and 73% at
  0.30, where the green dot marks the target harvest.

**The red curve.** The bass fall steadily as the harvest rate rises: 500,000
with no harvest, about 286,000 at the modest harvest, about 100,000 at the
target harvest, and almost none at 0.40 per year.

**What to notice.**

- **The threshold was predicted, not discovered.** Cell 04 computed $H_c$
  from the equations before any simulation ran. The sweep confirms it.
- **More salmon costs bass.** The two curves move in opposite directions.
  The far right of the plot has the most salmon, but there the bass are
  nearly wiped out, which is the kind of overfishing that led DFO to close
  the bass fishery in 1996. A workable policy sits between the tipping point and that extreme.

---
### Connecting the threshold to real management

- **DFO's reference point.** DFO's limit reference point for bass is 330,000
  spawners, above the model's critical level of 250,000. A policy that keeps
  the bass above 330,000 cannot reach the recovery side of the threshold.
- **The 100,000 target.** The Atlantic Salmon Federation and other groups ask
  for about 100,000 bass spawners, citing DFO work suggesting that about half
  of the smolts would then survive the trip through the estuary. In the model
  that is $H \approx 0.30$.
- **The existing quota.** A **quota** is the largest number of fish that may
  be caught in a year. DFO gave First Nations fishers a bass quota of 50,000
  fish in 2018 and raised it to 175,000 in 2024, but that many bass have
  never actually been caught. Taking 175,000 of 500,000 bass is 35% a year,
  or $H = -\ln(1 - 0.35) \approx 0.43$, well above $H_c$.

A fixed quota is not the same as a fixed rate $H$. As the bass decline, a
fixed catch of 175,000 becomes a larger and larger *fraction* of the bass
that are left, which can wipe out the bass just as overfishing did before
1996. That is one reason the people who set fishing rules prefer limits that
shrink when the fish population shrinks.

---
### What this model teaches, and what it leaves out

- **Generalist predators break the cycle.** When a predator has other food,
  the prey's collapse no longer starves it, so the feedback that drives
  Lotka-Volterra cycles is cut.
- **Depensation makes collapse self-reinforcing.** Because each bass's catch
  levels off, the risk to each salmon rises as the run shrinks.
- **Recovery has a threshold.** Harvest below $H_c$ changes almost nothing.
  Above it, the salmon recover steadily. Equilibrium analysis found the
  threshold before any simulation was run.

The model is deliberately simple. It leaves out:

- **Climate.** Warming water in the Gulf of St. Lawrence helps more young
  bass survive to adulthood, and it has let bass spread north. Large schools
  were first seen off southern Labrador, hundreds of kilometers north of the
  Miramichi, in 2017 and returned in 2019 and 2023, raising concern for the
  salmon, trout and Arctic char there. In this model that would mean $L$
  growing with time. Warmer rivers also stress salmon directly, because
  salmon need cold water.
- **Alternate prey.** Gaspereau, smelt and shad, small fish that bass also
  eat, have declined as the bass grew. A generalist can eat down
  its other foods too, which would make $L$ shrink over time instead of
  staying fixed.
- **Other causes of decline.** DFO points to conditions in the ocean, warming
  water, other predators, and even the weight of the electronic tags that
  researchers attach to smolts to follow them. The model lumps all of these
  into a constant marine mortality $m$.
- **Life stages.** Real salmon spend years in the river and at sea, so the
  response to any change is delayed.
- **Uncertainty.** These parameters were chosen to follow the real timeline,
  not estimated from data. DFO's own 2022 analysis concluded it is not clear
  that reducing bass to early-2000s levels would improve smolt survival, while
  the Atlantic Salmon Federation, Miramichi groups and a group of anglers who
  are suing DFO argue that it would. Fitting a model like this one to counts
  from the river, as the Stan case study does for lynx and hares, is how
  scientists try to settle such disagreements.

---
### Sources

- John Ritter (2024), *Is Striped Bass Predation the Cause of the Decline in
  and Low Atlantic Salmon Returns to the Miramichi River?* Bass counts, salmon
  returns in other Gulf rivers, the 2023 tagging results.
  <https://www.bradburnsfishing.com/wp-content/uploads/2024/05/Assessment-of-Miramichi-Salmon-Decline-20242.pdf>
- Rip Cunningham (2025), "Striped Bass Returned to this River. Now, They're
  Decimating the Salmon Population.", *Wired2Fish*. The 93% decline, the 1996
  closure, DFO's 330,000 reference point.
  <https://www.wired2fish.com/opinions-philosophies/striped-bass-and-salmon-miramichi-river>
- Jacques Poitras (2026), "'Our river is almost dead': What's killing an
  iconic Miramichi fishery?", *CBC News*. Salmon returns in 2025, the First
  Nations quota, the lawsuit, DFO's position.
  <https://www.cbc.ca/news/canada/new-brunswick/miramichi-atlantic-salmon-striped-bass-9.7373900>
- "Kill more striped bass to help wild salmon survive in Miramichi River:
  Atlantic Salmon Federation" (2024), *PNI Atlantic News*. Smolt survival then
  and now, the 100,000 target.
  <https://www.saltwire.com/atlantic-canada/kill-more-striped-bass-to-help-wild-salmon-survive-in-miramichi-river-atlantic-salmon-federation-100931947>
- Gord Follett (2026), "Concerns persist over 'unexpected' presence of striped
  bass off Labrador", *CBC News*. The northward range expansion.
  <https://www.cbc.ca/news/canada/newfoundland-labrador/concerns-persist-over-unexpected-presence-of-striped-bass-off-labrador-9.7236986>
- Robichaud-LeBlanc, Courtenay and Locke (1996), "Spawning and early life
  history of a northern population of striped bass (*Morone saxatilis*) in the
  Miramichi River estuary, Gulf of St. Lawrence", *Canadian Journal of
  Zoology* 74: 1645-1655. When and where the bass spawn.
  <https://doi.org/10.1139/z96-182>
- Bob Carpenter (2018), *Predator-Prey Population Dynamics: the Lotka-Volterra
  model in Stan*. The lynx and hare fit.
  <https://mc-stan.org/learn-stan/case-studies/lotka-volterra-predator-prey.html>
- Wusu and Olabanjo (2023), "Dynamics of Sustainable Fisheries: A Mathematical
  Approach using Lotka-Volterra Equations", *Preprints.org*. The textbook
  model with self-competition terms.
  <https://www.preprints.org/manuscript/202309.1296/v1>
- Christensen and Walters (2005), "Using ecosystem modeling for fisheries
  management: Where are we?", ICES CM 2005/M:19. Background on ecosystem
  models used in fisheries policy.
  <https://ices-library.figshare.com/ndownloader/files/44892622>